In [1]:
import subprocess
import re
import pandas as pd
import shutil
from tqdm import tqdm
import openpyxl
import os
import gc

In [2]:
# quartus_bin = r"C:\altera_standard\25.1std\quartus\bin64"

# # Compile
# subprocess.run(
#     [f"{quartus_bin}\\quartus_sh.exe", "--flow", "compile", "generic_multiplier"],
#         cwd=rf"C:\Users\Rafa\Desktop\HCR\generic_multiplier",
#     check=True
# )

# # Timing
# subprocess.run(
#         [f"{quartus_bin}\\quartus_sta.exe", "-t", "script.tcl"],
#         cwd=rf"C:\Users\Rafa\Desktop\HCR\generic_multiplier",
#     check=True
# )

## Diretorios

In [3]:
# version = "generic_multiplier_sqr"
version = "generic_multiplier"

quartus_bin = r"C:\altera_standard\25.1std\quartus\bin64"
base_dir = rf"C:\Users\Rafa\Desktop\HCR\Adder-Tree-Analysis\{version}"

n_values = [16, 24, 32, 48, 64, 96, 128, 256, 512] 


## Functions

In [4]:
# =========================
# altera n no VHDL
# =========================
def set_n(vhdl_file, n):
    with open(vhdl_file, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    text = re.sub(
        r"n\s*:\s*integer\s*:=\s*\d+",
        f"n : integer := {n}",
        text
    )

    with open(vhdl_file, "w", encoding="utf-8") as f:
        f.write(text)


# =========================
# acha nome do projeto .qpf
# =========================
def find_project_name(project_dir):
    qpf_files = [f for f in os.listdir(project_dir) if f.endswith(".qpf")]

    if not qpf_files:
        raise FileNotFoundError(f"Nenhum arquivo .qpf encontrado em {project_dir}")

    return os.path.splitext(qpf_files[0])[0]


# =========================
# roda quartus
# =========================
def run_quartus(project_dir):
    shutil.rmtree(os.path.join(project_dir, "db"), ignore_errors=True)
    shutil.rmtree(os.path.join(project_dir, "incremental_db"), ignore_errors=True)

    project_name = find_project_name(project_dir)

    try:
        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sh.exe"),
                "--flow",
                "compile",
                project_name
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )

        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sta.exe"),
                "-t",
                "script.tcl"
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"Quartus failed in {project_dir} with return code {e.returncode}\n"
            f"stdout:\n{e.stdout}\n"
            f"stderr:\n{e.stderr}"
        ) from e


# =========================
# parse dos paths
# =========================
def parse_paths(file):
    paths = []
    current = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Path #\d+: Delay is\s+([\d\.]+)", line)
            if m:
                if current:
                    paths.append(current)

                current = {
                    "delay": float(m.group(1)),
                    "ic": None,
                    "cell": None
                }
                continue

            if current is None:
                continue

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["ic"] = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["cell"] = float(m.group(1))

    if current:
        paths.append(current)

    df = pd.DataFrame(paths)

    if df.empty:
        return df

    return df.dropna()


# =========================
# calcula métricas
# =========================
def compute_metrics(df):
    critical = df.loc[df["delay"].idxmax()]

    return {
        "Delay_CP": critical["delay"],
        "IC_CP": critical["ic"],
        "CELL_CP": critical["cell"],

        "Delay_MAX": df["delay"].max(),
        "IC_MAX": df["ic"].max(),
        "CELL_MAX": df["cell"].max(),

        "Delay_MEAN": df["delay"].mean(),
        "IC_MEAN": df["ic"].mean(),
        "CELL_MEAN": df["cell"].mean()
    }


# =========================
# extrai caminho crítico
# =========================
def extract_critical_path(file):
    delay = None
    ic = None
    cell = None
    from_node = None
    to_node = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Delay is\s+([\d\.]+)", line)
            if m:
                delay = float(m.group(1))

            m = re.search(r";\s*From Node\s*;\s*(.*?)\s*;", line)
            if m:
                from_node = m.group(1)

            m = re.search(r";\s*To Node\s*;\s*(.*?)\s*;", line)
            if m:
                to_node = m.group(1)

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                ic = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                cell = float(m.group(1))

    return {
        "delay": delay,
        "ic": ic,
        "cell": cell,
        "from": from_node,
        "to": to_node
    }


In [5]:
# =========================
# loop principal
# =========================
resultados = []


project_dir = os.path.join(base_dir)
vhdl_file = os.path.join(project_dir, f"generic_multiplier.vhd")
report_file = os.path.join(project_dir, "io_paths.txt")
worst_file = os.path.join(project_dir, "worst_path.txt")

resultados_m = []

print(f"\n==============================")
print(f" Rodando projeto com vetores")
print(f"==============================")

for n in tqdm(n_values):

    df = None
    metrics = None
    cp = None

    try:
        print(f"\nRodando n = {n}")

        set_n(vhdl_file, n)
        run_quartus(project_dir)

        df = parse_paths(report_file)

        if df.empty:
            raise ValueError("Nenhum path encontrado em io_paths.txt")

        metrics = compute_metrics(df)
        cp = extract_critical_path(worst_file)

        linha = {
            "n": n,

            "Delay_CP": cp["delay"],
            "IC_CP": cp["ic"],
            "CELL_CP": cp["cell"],

            "From_Node": cp["from"],
            "To_Node": cp["to"],

            "Delay_MAX": metrics["Delay_MAX"],
            "IC_MAX": metrics["IC_MAX"],
            "CELL_MAX": metrics["CELL_MAX"],

            "Delay_MEAN": metrics["Delay_MEAN"],
            "IC_MEAN": metrics["IC_MEAN"],
            "CELL_MEAN": metrics["CELL_MEAN"],
        }

        resultados.append(linha)
        resultados_m.append(linha)

    except Exception as e:
        print(f"Erro em n={n}: {e}")
        continue

    finally:
        for name in ("df", "metrics", "cp"):
            globals().pop(name, None)
        gc.collect()

# salva um arquivo separado para cada número de vetores
df_m = pd.DataFrame(resultados_m).sort_values("n")

csv_m = os.path.join(base_dir, f"timing_n{n}.csv")
xlsx_m = os.path.join(base_dir, f"timing_n{n}.xlsx")

df_m.to_csv(csv_m, index=False)
df_m.to_excel(xlsx_m, index=False)

print(f"\nArquivos salvos para n = {n}:")
print(csv_m)
print(xlsx_m)


 Rodando projeto com vetores


  0%|          | 0/9 [00:00<?, ?it/s]


Rodando n = 16


 11%|█         | 1/9 [00:56<07:28, 56.09s/it]


Rodando n = 24


...


Rodando n = 256


 89%|████████▉ | 8/9 [12:41<02:16, 136.53s/it]


Rodando n = 512


100%|██████████| 9/9 [41:46<00:00, 278.54s/it]


Arquivos salvos para n = 512:
C:\Users\Rafa\Desktop\HCR\Adder-Tree-Analysis\generic_multiplier\timing_n512.csv
C:\Users\Rafa\Desktop\HCR\Adder-Tree-Analysis\generic_multiplier\timing_n512.xlsx


In [6]:
# =========================

# salva resultado geral
# =========================
final_df = pd.DataFrame(resultados).sort_values(["n"])

csv_final = os.path.join(base_dir, "timing_all_vectors.csv")
xlsx_final = os.path.join(base_dir, "timing_all_vectors.xlsx")

final_df.to_csv(csv_final, index=False)
final_df.to_excel(xlsx_final, index=False)

print("\nResultado final:")
print(final_df)

print("\nArquivos gerais salvos:")
print(csv_final)
print(xlsx_final)


Resultado final:
     n  Delay_CP   IC_CP  CELL_CP From_Node    To_Node  Delay_MAX  IC_MAX  \
0   16     7.144   2.547    4.597     a0[1]    sum[15]      7.144   2.547   
1   24     6.850   2.305    4.545    a1[13]     sum[7]      6.850   2.319   
2   32    10.800   5.338    5.462     a1[9]    sum[60]     10.800   6.307   
3   48    14.970   7.353    7.617    a1[29]    sum[95]     14.970   7.353   
4   64    19.195   8.543   10.652     a1[3]   sum[115]     19.195   9.001   
5   96    26.434   9.093   17.341    a1[92]   sum[191]     26.434  11.493   
6  128    34.123  15.745   18.378   a0[115]   sum[253]     34.123  16.163   
7  256    52.831  23.349   29.482   a0[228]   sum[474]     52.831  24.501   
8  512   119.830  75.425   44.405   a0[304]  sum[1023]    119.830  76.343   

   CELL_MAX  Delay_MEAN    IC_MEAN  CELL_MEAN  
0     4.648    5.859766   1.731109   4.128656  
1     4.596    6.023729   1.573969   4.449760  
2     6.780    9.688472   4.513484   5.174988  
3     7.617   14.44